<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/plantdoc_ResNet50_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 45.29 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import ResNet50, resnet
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

In [ ]:
img_size = (224, 224)   # ResNet50 expects 224x224
batch_size = 32

train_ds_full = image_dataset_from_directory(
    "/content/PlantDoc/train",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = image_dataset_from_directory(
    "/content/PlantDoc/test",
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds_full.class_names
num_classes = len(class_names)

# Split train into train + val (20%)
val_batches = tf.data.experimental.cardinality(train_ds_full) // 5
val_ds = train_ds_full.take(val_batches)
train_ds = train_ds_full.skip(val_batches)

print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 2342 files belonging to 28 classes.
Found 236 files belonging to 27 classes.
Classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf', 'Tomato two spotted spider mites leaf', 'grape leaf', 'grape leaf black rot']
Num Classes: 28


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

preprocess_input = resnet.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(data_augmentation(x)), y))
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds  = test_ds.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
base_model = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot   = val_ds.map(one_hot_map)
test_ds_onehot  = test_ds.map(one_hot_map)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=15
)

Epoch 1/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 71s 816ms/step - accuracy: 0.1202 - loss: 4.1705 - val_accuracy: 0.3594 - val_loss: 2.9785
Epoch 2/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 729ms/step - accuracy: 0.2631 - loss: 3.1394 - val_accuracy: 0.4375 - val_loss: 2.6604
Epoch 3/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 717ms/step - accuracy: 0.3537 - loss: 2.8767 - val_accuracy: 0.4866 - val_loss: 2.4717
Epoch 4/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 49s 737ms/step - accuracy: 0.3996 - loss: 2.7339 - val_accuracy: 0.5156 - val_loss: 2.3643
Epoch 5/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 712ms/step - accuracy: 0.4394 - loss: 2.6210 - val_accuracy: 0.5179 - val_loss: 2.3045
Epoch 6/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 719ms/step - accuracy: 0.4589 - loss: 2.5326 - val_accuracy: 0.5446 - val_loss: 2.2531
Epoch 7/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 52s 788ms/step - accuracy: 0.4751 - loss: 2.4520 - val_accuracy: 0.5558 - val_loss: 2.1966
Epoch 8/15
60/60 ━━━━━━━━━━━━━━━━━━━━ 78s 729ms/step - accuracy: 0.4728 - loss: 2.3722 - val_accu

In [ ]:
base_model.trainable = True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=40,
    initial_epoch=15
)

Epoch 16/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 98s 975ms/step - accuracy: 0.4526 - loss: 2.5606 - val_accuracy: 0.5982 - val_loss: 2.0057
Epoch 17/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 55s 820ms/step - accuracy: 0.5017 - loss: 2.3568 - val_accuracy: 0.5871 - val_loss: 2.0672
Epoch 18/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 54s 823ms/step - accuracy: 0.5446 - loss: 2.2487 - val_accuracy: 0.6295 - val_loss: 1.9714
Epoch 19/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 712ms/step - accuracy: 0.5676 - loss: 2.1784 - val_accuracy: 0.6295 - val_loss: 1.9798
Epoch 20/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 731ms/step - accuracy: 0.5848 - loss: 2.1293 - val_accuracy: 0.6339 - val_loss: 1.9585
Epoch 21/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 719ms/step - accuracy: 0.6344 - loss: 2.1037 - val_accuracy: 0.6429 - val_loss: 1.9340
Epoch 22/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 731ms/step - accuracy: 0.6252 - loss: 2.0225 - val_accuracy: 0.6384 - val_loss: 1.9261
Epoch 23/40
60/60 ━━━━━━━━━━━━━━━━━━━━ 48s 733ms/step - accuracy: 0.6442 - loss: 2.0000 - 

In [ ]:
TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    all_probs, all_labels = [], []
    for images, labels in dataset:
        images = tf.cast(images, tf.float32)
        logits = model(images, training=False)
        scaled_logits = logits / T
        probs = tf.nn.softmax(scaled_logits).numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_probs)

y_true, y_probs = predict_with_temperature(model, test_ds_onehot, T=TEMPERATURE)
y_pred = np.argmax(y_probs, axis=1)

y_true_int = np.argmax(y_true, axis=1)

present_classes = np.unique(y_true_int)
if len(present_classes) > 1:
    y_true_bin_present = label_binarize(y_true_int, classes=present_classes)
    y_probs_present = y_probs[:, present_classes]
    auc = roc_auc_score(
        y_true_bin_present,
        y_probs_present,
        average="macro",
        multi_class="ovr"
    )
else:
    print("⚠️ Only one class present in test set, AUC not defined.")
    auc = np.nan

# Other metrics
accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true_int, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true_int, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true_int, y_pred)

print("\n Evaluation Metrics After Calibration:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics After Calibration:
Accuracy : 0.4534
Precision: 0.4630
Recall   : 0.4534
F1 Score : 0.4382
AUC      : 0.9235
Kappa    : 0.4331


In [ ]:
model.save("model.h5")
print("Model saved successfully!")

Model saved successfully!


In [ ]:
from google.colab import files

files.download("model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>